In [2]:
import polars as pl
from ml4t.engineer import compute_features
import pandas as pd

C:\Users\thoma\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ml4t\engineer\features\ml\__init__.py:9: UserWarning: Feature 'cyclical_encode': lookback=0 but has period/window parameter. Consider using lookback='period' or specifying the actual lookback.
  from ml4t.engineer.features.ml.cyclical_encode import *  # noqa: F403


In [13]:
prices = pl.read_csv("cleaned_prices.csv")

In [15]:
macro = pl.read_csv("cleaned_macro.csv")

In [28]:
# Updated list of features to compute
# Bollinger Bands provide: 'upper', 'lower', and 'middle' (which is the SMA)
feature_list = ["rsi", "macd", "bollinger_bands", "sma"]

all_ticker_features = []

for ticker in tickers:
    # 1. Setup the ticker dataframe with spoofed OHLCV
    df_ticker = prices.select([
        pl.col("date"),
        pl.col(ticker).alias("close")
    ]).with_columns([
        pl.col("close").alias("high"),
        pl.col("close").alias("low"),
        pl.col("close").alias("open"),
        pl.lit(1.0).alias("volume")
    ])
    
    # 2. Compute the new expanded feature set
    # Note: 'sma' usually defaults to a 20-period window in many libraries
    features = compute_features(df_ticker, feature_list)
    
    # 3. Create 'Distance from SMA' - a very common RL feature
    # This tells the model if the price is currently "stretched" far from its average
    features = features.with_columns(
        ((pl.col("close") / pl.col("sma")) - 1).alias("dist_from_sma")
    )
    
    # 4. Rename with ticker prefix
    features = features.rename({
        col: f"{ticker}_{col}" for col in features.columns if col != "date"
    })
    
    all_ticker_features.append(features)

# Re-join everything
final_df = all_ticker_features[0]
for next_df in all_ticker_features[1:]:
    final_df = final_df.join(next_df, on="date", how="left")

final_df = final_df.join(macro, on="date", how="left")
final_df = final_df.drop_nulls()

print(final_df.head())

shape: (5, 55)
┌────────────┬────────────┬────────────┬────────────┬───┬──────────┬──────────┬────────┬──────────┐
│ date       ┆ SPY_close  ┆ SPY_high   ┆ SPY_low    ┆ … ┆ FEDFUNDS ┆ CPIAUCSL ┆ T10Y2Y ┆ INDPRO   │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---      ┆ ---      ┆ ---    ┆ ---      │
│ date       ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64      ┆ f64      ┆ f64    ┆ f64      │
╞════════════╪════════════╪════════════╪════════════╪═══╪══════════╪══════════╪════════╪══════════╡
│ 2007-09-30 ┆ 108.383331 ┆ 108.383331 ┆ 108.383331 ┆ … ┆ 4.94     ┆ 208.292  ┆ 0.62   ┆ 114.357  │
│ 2007-10-31 ┆ 109.853676 ┆ 109.853676 ┆ 109.853676 ┆ … ┆ 4.76     ┆ 208.903  ┆ 0.54   ┆ 113.9957 │
│ 2007-11-30 ┆ 105.598778 ┆ 105.598778 ┆ 105.598778 ┆ … ┆ 4.49     ┆ 210.565  ┆ 0.93   ┆ 113.9019 │
│ 2007-12-31 ┆ 104.409706 ┆ 104.409706 ┆ 104.409706 ┆ … ┆ 4.24     ┆ 211.16   ┆ 0.99   ┆ 113.9673 │
│ 2008-01-31 ┆ 98.096947  ┆ 98.096947  ┆ 98.096947  ┆ … ┆ 3.94     ┆ 212.516  ┆ 1.5  

In [29]:
final_df

date,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_bollinger_bands,SPY_macd,SPY_rsi,SPY_sma,SPY_dist_from_sma,EFA_close,EFA_high,EFA_low,EFA_open,EFA_volume,EFA_bollinger_bands,EFA_macd,EFA_rsi,EFA_sma,EFA_dist_from_sma,TLT_close,TLT_high,TLT_low,TLT_open,TLT_volume,TLT_bollinger_bands,TLT_macd,TLT_rsi,TLT_sma,TLT_dist_from_sma,VNQ_close,VNQ_high,VNQ_low,VNQ_open,VNQ_volume,VNQ_bollinger_bands,VNQ_macd,VNQ_rsi,VNQ_sma,VNQ_dist_from_sma,DBC_close,DBC_high,DBC_low,DBC_open,DBC_volume,DBC_bollinger_bands,DBC_macd,DBC_rsi,DBC_sma,DBC_dist_from_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO
date,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,f64,f64,f64
2007-09-30,108.383331,108.383331,108.383331,108.383331,1.0,"{111.091619,97.266317,83.441016}",NaN,75.091453,97.266317,0.114295,46.893764,46.893764,46.893764,46.893764,1.0,"{48.426663,40.742856,33.059049}",NaN,82.834274,40.742856,0.150969,50.324505,50.324505,50.324505,50.324505,1.0,"{51.524193,48.011154,44.498114}",NaN,57.286852,48.011154,0.048184,32.989189,32.989189,32.989189,32.989189,1.0,"{38.465246,32.316262,26.167278}",NaN,59.080686,32.316262,0.020823,22.830553,22.830553,22.830553,22.830553,1.0,"{22.121983,20.241359,18.360734}",NaN,72.026655,20.241359,0.127916,4.94,208.292,0.62,114.357
2007-10-31,109.853676,109.853676,109.853676,109.853676,1.0,"{112.558307,98.345082,84.131857}",NaN,76.353615,98.345082,0.117023,48.886707,48.886707,48.886707,48.886707,1.0,"{49.389794,41.45177,33.513747}",NaN,85.172545,41.45177,0.179364,51.238468,51.238468,51.238468,51.238468,1.0,"{51.949462,48.164577,44.379691}",NaN,60.298905,48.164577,0.063821,33.681652,33.681652,33.681652,33.681652,1.0,"{38.439855,32.601697,26.76354}",NaN,60.603198,32.601697,0.033126,24.771679,24.771679,24.771679,24.771679,1.0,"{23.128056,20.556759,17.985462}",NaN,77.712982,20.556759,0.205038,4.76,208.903,0.54,113.9957
2007-11-30,105.598778,105.598778,105.598778,105.598778,1.0,"{113.109542,99.138255,85.166968}",NaN,65.940645,99.138255,0.065167,47.115211,47.115211,47.115211,47.115211,1.0,"{49.907163,42.002604,34.098045}",NaN,75.347561,42.002604,0.121721,53.980396,53.980396,53.980396,53.980396,1.0,"{52.979214,48.565162,44.15111}",NaN,67.665671,48.565162,0.111505,30.491678,30.491678,30.491678,30.491678,1.0,"{38.386122,32.661687,26.937252}",NaN,51.159647,32.661687,-0.066439,24.617363,24.617363,24.617363,24.617363,1.0,"{23.854217,20.838678,17.823139}",NaN,76.383661,20.838678,0.18133,4.49,210.565,0.93,113.9019
2007-12-31,104.409706,104.409706,104.409706,104.409706,1.0,"{113.425748,99.815298,86.204848}",NaN,63.340888,99.815298,0.046029,45.710583,45.710583,45.710583,45.710583,1.0,"{50.214974,42.396741,34.578509}",NaN,68.591287,42.396741,0.078163,53.641533,53.641533,53.641533,53.641533,1.0,"{53.578931,49.012068,44.445205}",NaN,66.03487,49.012068,0.094456,28.841171,28.841171,28.841171,28.841171,1.0,"{38.334582,32.688962,27.043343}",NaN,47.072485,32.688962,-0.117709,26.267769,26.267769,26.267769,26.267769,1.0,"{24.954627,21.136525,17.318422}",NaN,80.270652,21.136525,0.242767,4.24,211.16,0.99,113.9673
2008-01-31,98.096947,98.096947,98.096947,98.096947,1.0,"{112.865083,100.313558,87.762033}",NaN,51.689498,100.313558,-0.022097,42.1236,42.1236,42.1236,42.1236,1.0,"{50.003303,42.683815,35.364328}",NaN,55.022756,42.683815,-0.013125,54.765831,54.765831,54.765831,54.765831,1.0,"{54.271953,49.519455,44.766956}",NaN,68.727842,49.519455,0.105946,28.615925,28.615925,28.615925,28.615925,1.0,"{38.207663,32.743144,27.278624}",NaN,46.526203,32.743144,-0.126048,27.033499,27.033499,27.033499,27.033499,1.0,"{26.048964,21.476622,16.90428}",NaN,81.769855,21.476622,0.258741,3.94,212.516,1.5,114.2334
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-07-31,539.458191,539.458191,539.458191,539.458191,1.0,"{549.720528,445.7